# Modern CNN Architectures

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/cnns/04-modern-architectures

From-scratch parameter counting, bottleneck blocks, and residual connections — the building blocks of every modern vision model.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(0)

## Intuition — the tricks that made CNNs deep and efficient

Plain CNN stacks stop improving past a certain depth — gradients vanish and parameters explode. A
handful of architectural ideas fixed this and define every modern vision model. **1×1 convolutions**
mix channels cheaply (dimensionality reduction). **Bottleneck blocks** (reduce → process → expand)
cut computation. **Residual/skip connections** (`y = x + f(x)`) give gradients a shortcut so networks
can be *hundreds* of layers deep. **Compound scaling** (EfficientNet) balances width, depth, and
resolution. **Depthwise separable convolutions** (MobileNet) slash parameters for mobile deployment.
We implement each and verify the parameter math.

## 1×1 convolutions: channel mixing without spatial mixing

A 1×1 conv applies the same linear projection to every spatial position independently.
It is the primary tool for reducing or expanding channel counts cheaply.

In [ ]:
def conv1x1_params(c_in, c_out):
    """Parameters of a 1×1 convolution (no bias)."""
    return 1 * 1 * c_in * c_out

def conv3x3_params(c_in, c_out):
    """Parameters of a 3×3 convolution (no bias)."""
    return 3 * 3 * c_in * c_out

# Standard 3×3: 256 → 256
standard = conv3x3_params(256, 256)
print(f'Standard 3×3 (256→256): {standard:,} params')

# Bottleneck: 1×1 (256→64) + 3×3 (64→64) + 1×1 (64→256)
bottleneck = conv1x1_params(256, 64) + conv3x3_params(64, 64) + conv1x1_params(64, 256)
print(f'Bottleneck  (256→64→64→256): {bottleneck:,} params')
print(f'Reduction factor: {standard/bottleneck:.1f}×')

assert standard == 589_824
assert bottleneck == 69_632
print(f'VERIFY: bottleneck uses {standard//bottleneck}× fewer parameters.')

**What to notice:** a **1×1 convolution** touches one pixel at a time but mixes across *channels* —
it's a per-pixel fully-connected layer. That makes it a cheap way to **reduce channel dimensions**
before an expensive 3×3 conv (the bottleneck idea) or to add non-linear channel mixing, at a fraction
of a full conv's cost.

## Bottleneck residual block — numpy implementation

Implement the forward pass (no training, just shape verification).
The skip connection adds the input back before the final ReLU.

In [ ]:
def relu(x): return np.maximum(0, x)

def layernorm_simple(x):
    """Simplified normalisation: standardise over the channel axis."""
    mu  = x.mean(axis=-1, keepdims=True)
    std = x.std(axis=-1, keepdims=True) + 1e-5
    return (x - mu) / std

def conv_nd(x, W):
    """Simple matrix-multiply proxy for a 1×1 conv on a batch of vectors.
    x: (N, C_in), W: (C_in, C_out) → output: (N, C_out)."""
    return x @ W

np.random.seed(7)
N, C = 8, 256   # batch of 8 spatial positions, 256 channels
x = np.random.randn(N, C) * 0.1

# Bottleneck weights
scale = 0.1
W_down = np.random.randn(256, 64)  * scale
W_3x3  = np.random.randn(64, 64)   * scale
W_up   = np.random.randn(64, 256)  * scale

def bottleneck_block(x, W_down, W_3x3, W_up):
    """Bottleneck residual block (1×1 → 3×3 → 1×1 + skip)."""
    h = relu(layernorm_simple(conv_nd(x, W_down)))   # 256 → 64
    h = relu(layernorm_simple(conv_nd(h, W_3x3)))    # 64  → 64
    h = layernorm_simple(conv_nd(h, W_up))           # 64  → 256
    return relu(x + h)                               # skip + ReLU

out = bottleneck_block(x, W_down, W_3x3, W_up)
print('Input shape:', x.shape, '→ Output shape:', out.shape)
assert out.shape == x.shape, 'Bottleneck must preserve shape'
print('VERIFY: bottleneck block input == output shape.')

**What to notice:** the **bottleneck block** wraps the expensive 3×3 conv between two 1×1 convs
that first *reduce* then *restore* the channel count — so the 3×3 operates on far fewer channels. This
cuts parameters and FLOPs dramatically versus a plain block, which is how ResNet-50/101/152 stay
affordable at great depth.

## Skip connections and gradient flow

With y = F(x) + x, the gradient ∂y/∂x = ∂F/∂x + 1.
The constant 1 is the skip path — gradients always flow regardless of F.

In [ ]:
# Compare activation norms through deep stacks: plain vs. residual
np.random.seed(1)
n_blocks = 30
d = 64

plain_norms = []
residual_norms = []

x_plain = np.random.randn(1, d)
x_res   = np.random.randn(1, d)

for _ in range(n_blocks):
    W = np.random.randn(d, d) * (0.1 / np.sqrt(d))
    # Plain: x → tanh(Wx)
    x_plain = np.tanh(x_plain @ W)
    plain_norms.append(np.linalg.norm(x_plain))
    # Residual: x → x + tanh(Wx)
    x_res = x_res + np.tanh(x_res @ W)
    residual_norms.append(np.linalg.norm(x_res))

fig, ax = plt.subplots()
ax.plot(plain_norms, color='#f59e0b', label='plain (tanh layers)')
ax.plot(residual_norms, color='#6366f1', label='residual')
ax.set_xlabel('block depth'); ax.set_ylabel('||activations||')
ax.set_title('Activation norm through 30 blocks: plain vs. residual')
ax.legend(); plt.show()

print('Final plain norm:', round(plain_norms[-1], 4))
print('Final residual norm:', round(residual_norms[-1], 4))

**What to notice:** through a deep **plain** stack the activation norm drifts (toward vanish or
explosion), but with **skip connections** it stays stable — because `y = x + f(x)` lets the signal
(and its gradient) flow through the identity path untouched. This is the single idea that unlocked
100+ layer networks; without it, deep plain nets are untrainable.

## EfficientNet compound scaling

Compound scaling jointly increases depth $d=\alpha^\phi$, width $w=\beta^\phi$, resolution $r=\gamma^\phi$
subject to $\alpha \cdot \beta^2 \cdot \gamma^2 \approx 2$ (doubling FLOPs per $\phi$ step).

In [ ]:
# EfficientNet compound scaling: FLOPs grow as w^2 * d * r^2
# (channels^2 for conv, linear in depth, resolution^2 for spatial)
def flops_scale(alpha, beta, gamma, phi):
    d = alpha ** phi
    w = beta  ** phi
    r = gamma ** phi
    return w**2 * d * r**2   # proportional to total FLOPs

# EfficientNet-found constants
alpha, beta, gamma = 1.2, 1.1, 1.15

print('phi | depth | width | res  | rel. FLOPs')
base = flops_scale(alpha, beta, gamma, 0)
for phi in range(8):
    f = flops_scale(alpha, beta, gamma, phi)
    print(f'{phi}   | {alpha**phi:.2f}  | {beta**phi:.2f}  | {gamma**phi:.2f} | {f/base:.1f}×')

# Verify: each step approximately doubles FLOPs
ratios = [flops_scale(alpha, beta, gamma, phi+1) / flops_scale(alpha, beta, gamma, phi)
          for phi in range(6)]
assert all(abs(r - 2.0) < 0.2 for r in ratios), 'Each phi step should ≈ double FLOPs'
print('VERIFY: each compound scaling step doubles FLOPs.')

**What to notice:** EfficientNet's **compound scaling** grows width, depth, and resolution
*together* by a balanced factor rather than cranking one alone — FLOPs scale as `w²·d·r²`. Balancing
the three gives better accuracy per FLOP than scaling any single dimension, the insight behind the
EfficientNet family.

## Depthwise separable convolutions

MobileNet factorizes standard convolution into depthwise (spatial) + pointwise (channel) steps.
This reduces parameters and FLOPs by ~8× for typical channel counts.

In [ ]:
def compare_params(K, C_in, C_out):
    """Compare standard vs. depthwise-separable conv parameters."""
    standard   = K * K * C_in * C_out
    depthwise  = K * K * C_in           # one K×K filter per input channel
    pointwise  = 1 * 1 * C_in * C_out  # 1×1 to mix channels
    dw_sep_total = depthwise + pointwise
    return standard, dw_sep_total, standard / dw_sep_total

print(f'{"Channels":>8} | {"Standard":>12} | {"DW-Sep":>10} | {"Reduction":>10}')
for C in [32, 64, 128, 256, 512]:
    std, dws, ratio = compare_params(K=3, C_in=C, C_out=C)
    print(f'{C:>8} | {std:>12,} | {dws:>10,} | {ratio:>9.1f}×')

# For large C, reduction approaches K^2 = 9 (for K=3)
std, dws, ratio = compare_params(K=3, C_in=1000, C_out=1000)
assert ratio > 8.5, 'For large C, DW-sep should give >8.5× reduction'
print('VERIFY: depthwise separable ≈ 9× fewer params for large C.')

**What to notice:** a **depthwise separable** convolution factors a standard conv into a
per-channel spatial filter (depthwise) plus a 1×1 channel mix (pointwise), using a small fraction of
the parameters. That factorization is what makes MobileNet small and fast enough for phones.

## The library way — verify the depthwise-separable savings

MobileNet's core claim has an exact closed form: a depthwise separable conv uses a fraction
`1/C_out + 1/K²` of a standard conv's parameters. The cell verifies our from-scratch counts match that
formula (the same layers you'd get from `torch.nn.Conv2d(..., groups=C_in)` + a 1×1 conv).

In [ ]:
K, C_in, C_out = 3, 64, 128

standard  = K * K * C_in * C_out                  # one dense K x K conv
depthwise = K * K * C_in                          # one K x K filter per input channel
pointwise = C_in * C_out                          # 1x1 mixing across channels
separable = depthwise + pointwise

ratio_measured  = separable / standard
ratio_formula   = 1 / C_out + 1 / (K * K)
print(f'standard conv params : {standard:,}')
print(f'separable conv params: {separable:,}  ({100*ratio_measured:.1f}% of standard)')
print(f'ratio: measured {ratio_measured:.4f}  vs  formula (1/C_out + 1/K^2) {ratio_formula:.4f}')
assert np.isclose(ratio_measured, ratio_formula), "separable savings must match the MobileNet formula"
print('depthwise-separable parameter savings verified ✓')

**What to notice:** the separable conv uses only ~**12%** of the standard conv's parameters, and
the measured ratio matches the theoretical `1/C_out + 1/K²` exactly. For a 3×3 kernel the `1/K² = 1/9`
term dominates — depthwise separable convs are ~8–9× cheaper, the whole reason MobileNet runs on
phones.

## Gotchas & tradeoffs

- **Skip connections need matching dimensions.** When `f(x)` changes channels/resolution, the shortcut
  needs a 1×1 conv (a "projection shortcut") to align shapes before adding.
- **Depth still needs normalization.** Residuals help, but very deep nets also rely on BatchNorm/LayerNorm
  to train stably — the two work together.
- **Efficiency trades a little accuracy.** Depthwise separable convs and aggressive bottlenecks save
  compute but can cost a bit of accuracy vs a full conv — a deployment tradeoff.
- **Scaling has diminishing returns.** Doubling FLOPs doesn't double accuracy; compound scaling
  mitigates but can't escape the plateau.

In [ ]:
# Skip connections keep gradients alive: d/dx of (x + f(x)) always includes the identity
# y = x + W2 @ relu(W1 @ x); the +I term means the gradient can't fully vanish
np.random.seed(0)
d = 8
W1 = np.random.randn(d, d) * 0.01     # tiny weights -> f(x) gradient is tiny
W2 = np.random.randn(d, d) * 0.01
x = np.random.randn(d)

# Jacobian of the RESIDUAL block includes + I; of the PLAIN block does not
relu_mask = (W1 @ x > 0).astype(float)
J_f = W2 @ (relu_mask[:, None] * W1)            # d f(x) / dx  (tiny)
J_plain = J_f
J_residual = np.eye(d) + J_f
print(f'plain    block ||Jacobian|| = {np.linalg.norm(J_plain):.4f}  (near 0 -> gradient vanishes)')
print(f'residual block ||Jacobian|| = {np.linalg.norm(J_residual):.4f}  (>= 1 thanks to +I)')

**What to notice:** with tiny weights the plain block's Jacobian norm is ~0 (the gradient would
vanish through many such layers), but the residual block's Jacobian is `I + J_f`, so its norm stays
`≥ 1` — the gradient always has the identity shortcut to flow through. That `+I` is, mathematically,
why ResNets train at depths that break plain networks.

## Key takeaways

- **1×1 convolutions** mix channels without spatial processing — core tool for channel count control.
- **Bottleneck block** (1×1 → 3×3 → 1×1) gives ~8-17× parameter savings vs. naive 3×3.
- **Skip connections**: gradient = ∂F/∂x + 1 — the +1 keeps gradients flowing regardless of depth.
- **MobileNet** depthwise separable conv: ~8× fewer params for the same receptive field.
- **EfficientNet** jointly scales depth, width, resolution to optimise the accuracy/FLOP tradeoff.

## ✏️ Your turn

### Exercise 1 — 1×1 convolution forward pass

A 1×1 convolution with weight matrix $W \in \mathbb{R}^{C_{in} \times C_{out}}$ applies the same linear projection to every spatial position. Implement it and verify: shape, and that it equals a per-position matrix multiply.

In [ ]:
import numpy as np

def conv1x1_forward(x, W):
    """1×1 convolution forward pass.
    x: (H, W, C_in) feature map, W: (C_in, C_out) weight matrix.
    Returns: (H, W, C_out) output."""
    # TODO(you): apply W to the channel dimension at every spatial position
    # Hint: reshape x to (H*W, C_in), apply W, reshape back
    ...

In [ ]:
np.random.seed(0)
H, W_size, C_in, C_out = 4, 4, 8, 3
x_feat = np.random.randn(H, W_size, C_in)
W_proj = np.random.randn(C_in, C_out)

out = conv1x1_forward(x_feat, W_proj)

assert out.shape == (H, W_size, C_out), \
    f"output shape must be ({H}, {W_size}, {C_out}), got {out.shape}"

# Spot-check: output at position (0,0) equals x[0,0] @ W
assert np.allclose(out[0, 0], x_feat[0, 0] @ W_proj), \
    "output at (0,0) must equal x[0,0] @ W"
assert np.allclose(out[2, 3], x_feat[2, 3] @ W_proj), \
    "output at (2,3) must equal x[2,3] @ W (same W at every position)"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def conv1x1_forward(x, W):
    H, W_size, C_in = x.shape
    return (x.reshape(-1, C_in) @ W).reshape(H, W_size, -1)
```

</details>

### Exercise 2 — Bottleneck parameter count

Compute the total number of parameters in a bottleneck residual block with channels $C_{in} = C_{out} = 256$ and bottleneck width $B = 64$. Compare to a plain two-layer 3×3 block.

In [ ]:
def bottleneck_params(c_in, c_out, b):
    """Parameter count for a bottleneck residual block (ignoring bias):
    1×1 (c_in→b) + 3×3 (b→b) + 1×1 (b→c_out)."""
    # TODO(you): sum the three layer parameter counts
    ...

def plain_block_params(c):
    """Parameter count for two stacked 3×3 convs (c→c→c), ignoring bias."""
    # TODO(you): two 3×3 convolutions
    ...

In [ ]:
bt = bottleneck_params(256, 256, 64)
pl = plain_block_params(256)

assert bt == 69_632, \
    f"bottleneck params should be 69,632, got {bt}"
assert pl == 1_179_648, \
    f"plain block params should be 1,179,648, got {pl}"
assert pl // bt >= 16, \
    "plain block should have at least 16× more parameters than bottleneck"
print(f"Bottleneck: {bt:,} params")
print(f"Plain:      {pl:,} params ({pl//bt}× more)")
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def bottleneck_params(c_in, c_out, b):
    return (1*1*c_in*b) + (3*3*b*b) + (1*1*b*c_out)

def plain_block_params(c):
    return 2 * (3*3*c*c)
```

</details>

### Exercise 3 — Residual block gradient

For y = F(x) + x, compute ∂L/∂x given ∂L/∂y (the upstream gradient).
Verify the skip path carries the gradient unchanged even when F's Jacobian is zero.

In [ ]:
import numpy as np

def residual_backward(grad_y, jac_F):
    """Backward pass of y = F(x) + x.
    grad_y: upstream gradient ∂L/∂y (1-D array).
    jac_F: Jacobian ∂F/∂x (2-D square matrix).
    Returns ∂L/∂x."""
    # TODO(you): ∂L/∂x = grad_y @ (jac_F + I)
    ...

In [ ]:
d = 4
grad_y = np.array([1.0, 2.0, 0.5, -1.0])
jac_F  = np.random.randn(d, d) * 0.1

grad_x = residual_backward(grad_y, jac_F)

assert grad_x.shape == grad_y.shape, "gradient shape must match input shape"
assert np.allclose(grad_x, grad_y @ (jac_F + np.eye(d))), \
    "∂L/∂x must equal grad_y @ (∂F/∂x + I)"

# Key property: when F's Jacobian is zero, skip path carries full gradient
zero_jac = np.zeros((d, d))
grad_x_dead = residual_backward(grad_y, zero_jac)
assert np.allclose(grad_x_dead, grad_y), \
    "with zero Jacobian (dead block), skip path must pass gradient unchanged"
print("✅ Exercise 3 passed")

<details>
<summary>💡 Show solution</summary>

```python
def residual_backward(grad_y, jac_F):
    return grad_y @ (jac_F + np.eye(len(grad_y)))
```

</details>

### Extra practice — DML #113: a residual block, forward pass

Exercise 3 derived the *gradient* of a generic skip connection $y = F(x) + x$.
This [Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem) problem asks for
the concrete forward pass: a 1-D input goes through two weight layers (matrix
multiplies) with a ReLU in between, then the **original input is added back**
before one final ReLU:

$$y = \text{ReLU}\big(W_2 \cdot \text{ReLU}(W_1 x) + x\big)$$

This is exactly the residual-block shape that motivated Exercise 3 — now you
implement the block those gradients flow through.

In [ ]:
def residual_block(x: np.ndarray, w1: np.ndarray, w2: np.ndarray) -> np.ndarray:
    """DML #113 -- a simple residual block: two linear layers + ReLUs + shortcut.
    x: (d,) input. w1, w2: (d, d) weight matrices.
    Returns: (d,) output, ReLU(w2 @ ReLU(w1 @ x) + x).
    """
    # TODO(you): first weight layer, then ReLU
    y = ...

    # TODO(you): second weight layer
    y = ...

    # TODO(you): add the shortcut (the original x), then the final ReLU
    y = ...

    return y

In [ ]:
# Checks — run me (DML's own test cases + edge cases)
w1 = np.array([[1.0, 0.0], [0.0, 1.0]])
w2 = np.array([[0.5, 0.0], [0.0, 0.5]])

assert np.allclose(residual_block(np.array([1.0, 2.0]), w1, w2), [1.5, 3.0])
assert np.allclose(residual_block(np.array([-1.0, 2.0]), w1, w2), [0.0, 3.0]), \
    "negative input to the first ReLU zeroes that branch; the shortcut still carries -1.0 through, but the final ReLU clips it"
assert np.allclose(residual_block(np.array([0.0, 0.0]), w1, w2), [0.0, 0.0]), "all-zero input -> all-zero output"

# Edge case: F(x) driven entirely negative by w1/w2 -- only the shortcut survives
w1_neg = np.array([[-1.0, 0.0], [0.0, -1.0]])
out = residual_block(np.array([2.0, 3.0]), w1_neg, w2)
assert np.allclose(out, [2.0, 3.0]), "ReLU(w1 @ x) is all zero here, so F(x) = 0 and y = ReLU(0 + x) = x"

# Edge case: larger dimensionality
rng = np.random.default_rng(0)
d = 5
x5 = rng.standard_normal(d)
w1_5, w2_5 = np.eye(d), np.eye(d)
out5 = residual_block(x5, w1_5, w2_5)
assert out5.shape == (d,)
assert np.allclose(out5, np.maximum(0, np.maximum(0, x5) + x5)), "with identity weights, y = ReLU(ReLU(x) + x)"
print("✅ DML #113 passed")

<details>
<summary>💡 Show solution</summary>

```python
def residual_block(x: np.ndarray, w1: np.ndarray, w2: np.ndarray) -> np.ndarray:
    y = np.dot(w1, x)
    y = np.maximum(0, y)
    y = np.dot(w2, y)
    y = y + x
    y = np.maximum(0, y)
    return y
```

</details>

### Extra practice — DML #137: a DenseNet dense block

ResNet's skip connection **adds** $F(x) + x$. DenseNet instead **concatenates**:
every layer's output is appended to a growing feature-map stack, and every later
layer convolves over *all* previously accumulated channels. With `growth_rate`
new channels added per layer, a block that starts with $C_0$ channels and runs
`num_layers` layers ends with $C_0 + \text{num\_layers} \times \text{growth\_rate}$
channels.

This [Deep-ML](https://github.com/Open-Deep-ML/DML-OpenProblem) problem operates
on a batched, multi-channel **NHWC** tensor (`(N, H, W, C)` — batch, height,
width, channels) rather than the single 2D image used earlier in this course.
Each iteration: ReLU the running feature stack, convolve (stride 1, no bias,
symmetric zero-padding so `H`/`W` never shrink) with that layer's kernel, then
concatenate the result onto the channel axis. `kernels[l]` has shape
`(kh, kw, C0 + l * growth_rate, growth_rate)` — its input-channel dimension
must match the *current* running channel count, which grows every iteration.
The batched, multi-channel convolution helper (`conv2d_nhwc`) is provided below;
your job is the dense-block loop itself.

In [ ]:
def conv2d_nhwc(x, kernel, padding=0):
    """Batched, multi-channel 'valid-after-padding' convolution, stride 1, no bias.
    x: (N, H, W, C_in). kernel: (kh, kw, C_in, C_out). Returns: (N, H', W', C_out).
    """
    if padding > 0:
        x = np.pad(x, ((0, 0), (padding, padding), (padding, padding), (0, 0)))
    N, H, W, c_in = x.shape
    kh, kw, k_cin, c_out = kernel.shape
    if k_cin != c_in:
        raise ValueError(f"kernel in-channels ({k_cin}) != feature-map channels ({c_in})")
    oh, ow = H - kh + 1, W - kw + 1
    out = np.zeros((N, oh, ow, c_out))
    for b in range(N):
        for i in range(oh):
            for j in range(ow):
                window = x[b, i:i + kh, j:j + kw, :]        # (kh, kw, C_in)
                for co in range(c_out):
                    out[b, i, j, co] = np.sum(window * kernel[:, :, :, co])
    return out

def dense_net_block(input_data, num_layers, growth_rate, kernels, kernel_size=(3, 3)):
    """DML #137 -- DenseNet dense block: concatenate each new conv's output onto
    the running feature-map stack, so every layer sees all previous channels.

    input_data: (N, H, W, C0). kernels: list of `num_layers` arrays, kernels[l]
    has shape (kh, kw, C0 + l*growth_rate, growth_rate).
    Returns: (N, H, W, C0 + num_layers*growth_rate).
    """
    kh, kw = kernel_size
    padding = (kh - 1) // 2   # 'same' padding: preserves H and W
    concatenated_features = input_data.copy().astype(float)

    for l in range(num_layers):
        # TODO(you): ReLU the running feature stack
        activated = ...

        # TODO(you): convolve with this layer's kernel (use conv2d_nhwc above)
        conv_output = ...

        # TODO(you): concatenate the new channels onto the running stack (channel axis = last, i.e. axis=3)
        concatenated_features = ...

    return concatenated_features

In [ ]:
# Checks — run me (DML's own test cases + edge cases)
np.random.seed(42)
X1 = np.random.randn(1, 1, 1, 2)
kernels1 = [np.random.randn(3, 3, 2 + i * 1, 1) * 0.01 for i in range(2)]
out1 = dense_net_block(X1, 2, 1, kernels1)
expected1 = np.array([[[[4.96714153e-01, -1.38264301e-01, -2.30186127e-03, -6.70426255e-05]]]])
assert out1.shape == (1, 1, 1, 4), "C0=2 + 2 layers * growth_rate=1 -> 4 channels"
assert np.allclose(out1, expected1, atol=1e-6)

np.random.seed(42)
X2 = np.random.randn(1, 2, 3, 2)
kernels2 = [np.random.randn(3, 3, 2 + i * 1, 1) * 0.01 for i in range(2)]
out2 = dense_net_block(X2, 2, 1, kernels2)
assert out2.shape == (1, 2, 3, 4), "spatial size (H, W) is preserved by the 'same' padding"

# Edge case: num_layers=0 -- the block is a no-op, output == input
out0 = dense_net_block(X1, 0, 1, [])
assert np.allclose(out0, X1), "zero layers: nothing to concatenate, output equals input"

# Edge case: growth_rate > 1 and multiple layers -- channel count grows correctly each step
rng = np.random.default_rng(0)
X3 = rng.standard_normal((2, 4, 4, 3))   # batch of 2, 3 input channels
kernels3 = [rng.standard_normal((3, 3, 3 + i * 2, 2)) * 0.01 for i in range(3)]  # growth_rate=2, 3 layers
out3 = dense_net_block(X3, 3, 2, kernels3)
assert out3.shape == (2, 4, 4, 3 + 3 * 2), "C0=3 + 3 layers * growth_rate=2 -> 9 channels"

# Edge case: mismatched kernel channels must raise, not silently misbehave
bad_kernel = [rng.standard_normal((3, 3, 99, 2))]
try:
    dense_net_block(X3, 1, 2, bad_kernel)
    raise AssertionError("expected a ValueError for mismatched kernel in-channels")
except ValueError:
    pass
print("✅ DML #137 passed")

<details>
<summary>💡 Show solution</summary>

```python
def dense_net_block(input_data, num_layers, growth_rate, kernels, kernel_size=(3, 3)):
    kh, kw = kernel_size
    padding = (kh - 1) // 2
    concatenated_features = input_data.copy().astype(float)
    for l in range(num_layers):
        activated = np.maximum(concatenated_features, 0.0)
        conv_output = conv2d_nhwc(activated, kernels[l], padding=padding)
        concatenated_features = np.concatenate([concatenated_features, conv_output], axis=3)
    return concatenated_features
```

</details>